# 🔬 Advanced Prompting Techniques

**Day 3 — Notebook 1 of 4 | Estimated Time: 30 minutes**

---

## What You'll Learn
- Self-consistency: sample multiple reasoning paths and take the majority answer
- Tree-of-thought: explore multiple solution branches
- Iterative refinement: improve outputs through multi-step prompting

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0"

In [ ]:
import sys, os
repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types
from IPython.display import Markdown

client = genai.Client(api_key=get_api_key())
MODEL_ID = "gemini-2.5-flash"
print("✅ Ready!")

---

## 1. Self-Consistency

**Idea:** Ask the model the same reasoning question multiple times with higher temperature, then take the **majority answer**. This reduces errors on complex reasoning tasks.

```
Run 1: Think step by step → Answer: 42
Run 2: Think step by step → Answer: 42
Run 3: Think step by step → Answer: 38
Run 4: Think step by step → Answer: 42
Run 5: Think step by step → Answer: 42

Majority answer: 42 ✓ (4 out of 5)
```

In [ ]:
from collections import Counter
import re

problem = """
A farmer has chickens and cows. He counts 30 heads and 80 legs total.
How many chickens does he have?

Think step by step. End your response with: ANSWER: [number]
"""

# Sample 5 reasoning paths
answers = []
for i in range(5):
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=problem,
        config=types.GenerateContentConfig(temperature=0.7),
    )
    text = response.text
    # Extract the final answer
    match = re.search(r'ANSWER:\s*(\d+)', text)
    if match:
        answer = int(match.group(1))
        answers.append(answer)
        print(f"Run {i+1}: Answer = {answer}")
    else:
        print(f"Run {i+1}: Could not extract answer")

# Majority vote
if answers:
    counter = Counter(answers)
    majority = counter.most_common(1)[0]
    print(f"\n🏆 Self-consistent answer: {majority[0]} (appeared {majority[1]}/{len(answers)} times)")

---

## 2. Tree-of-Thought (ToT)

**Idea:** Instead of one linear chain of thought, explore **multiple solution branches**, evaluate each, and pick the best one.

In [ ]:
prompt = """
Problem: A company needs to reduce costs by 20% without laying off employees.

Think about this using 3 different expert perspectives:

EXPERT 1 (Operations): Propose a solution focused on operational efficiency.
- Describe the approach
- List pros and cons
- Estimate savings

EXPERT 2 (Technology): Propose a solution focused on technology/automation.
- Describe the approach
- List pros and cons
- Estimate savings

EXPERT 3 (Finance): Propose a solution focused on financial restructuring.
- Describe the approach
- List pros and cons
- Estimate savings

FINAL VERDICT: After considering all three perspectives, which approach (or 
combination) would you recommend and why?
"""

response = client.models.generate_content(model=MODEL_ID, contents=prompt)
Markdown(response.text)

---

## 3. Iterative Refinement

**Idea:** Generate an initial output, then refine it through follow-up prompts.

In [ ]:
# Step 1: Generate initial draft
draft_prompt = "Write a product description for noise-cancelling wireless headphones priced at $199."
draft = client.models.generate_content(model=MODEL_ID, contents=draft_prompt)
print("📝 DRAFT:")
print(draft.text)
print()

In [ ]:
# Step 2: Critique the draft
critique_prompt = f"""Review this product description and list 3 specific improvements:

<draft>
{draft.text}
</draft>

For each improvement, explain WHY it would make the description more effective."""

critique = client.models.generate_content(model=MODEL_ID, contents=critique_prompt)
print("🔍 CRITIQUE:")
print(critique.text)
print()

In [ ]:
# Step 3: Apply improvements
refine_prompt = f"""Rewrite this product description by applying these improvements:

<original>
{draft.text}
</original>

<improvements>
{critique.text}
</improvements>

Write the improved version:"""

refined = client.models.generate_content(model=MODEL_ID, contents=refine_prompt)
print("✨ REFINED:")
Markdown(refined.text)

---

## 🏋️ Exercise 1: Self-Consistency on a Logic Puzzle

Use self-consistency (5 samples) to solve this logic puzzle:

In [ ]:
# Exercise 1: Self-consistency
# TODO: Run this 5 times and take the majority answer

puzzle = """
In a race, you overtake the person in second place. 
What place are you in now?

Think step by step. End with ANSWER: [your answer]
"""

# TODO: Implement self-consistency sampling
# Hint: Use the pattern from Section 1 above

answers = []
for i in range(5):
    pass  # Your code here

# Print majority answer

---

## 🏋️ Exercise 2: Iterative Refinement for Code

Use the 3-step iterative refinement pattern to improve a piece of code.

In [ ]:
# Exercise 2: Iterative code refinement
# TODO: Apply the draft → critique → refine pattern

# Step 1: Ask Gemini to write a function
task = "Write a Python function that validates an email address using regex."

# TODO: Step 1 - Generate draft
# TODO: Step 2 - Critique it (ask for security issues, edge cases, improvements)
# TODO: Step 3 - Apply improvements

# Your code here

---

## 📚 Further Reading

| Resource | Type | Link |
|----------|------|------|
| Self-Consistency Improves CoT Reasoning | Paper | [arxiv.org/abs/2203.11171](https://arxiv.org/abs/2203.11171) |
| Tree of Thoughts | Paper | [arxiv.org/abs/2305.10601](https://arxiv.org/abs/2305.10601) |
| Advanced Prompting Techniques | Video | [YouTube](https://www.youtube.com/watch?v=jC4v5AS4RIM) |

---

**Next up:** [02_Structured_Output_Generation.ipynb](./02_Structured_Output_Generation.ipynb)